### Lahore Air Quality & Smog Forecast

#### Notebook 4: Model Evaluation

Purpose

The purpose of this notebook is to evaluate the performance of the trained Random Forest models for forecasting PM2.5 concentrations one, two, and three days ahead. Using unseen test data, the models are assessed with standard regression metrics, including Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the coefficient of determination (R²). The evaluation provides an objective assessment of forecasting performance and identifies strengths and limitations before deployment in the forecasting website.


In [40]:
import pandas as pd
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [41]:
df = pd.read_csv("lahore_air_quality_features.csv")

In [42]:
df["date"] = pd.to_datetime(df["date"])
df.head()

,name,date,pm25,temperature,relativehumidity,wind_speed,wind_direction,lat,lon,location_id,is_smog_season,pm25_day1,pm25_day2,pm25_day3
0,"ARC, Lahore",2025-11-10,267.000000,17.669231,66.076923,1.215385,147.000000,31.522056,74.314944,6125629,1,267.708333,265.611111,234.470588
1,"ARC, Lahore",2025-11-11,267.708333,18.341667,64.875000,3.308333,208.625000,31.522056,74.314944,6125629,1,265.611111,234.470588,322.333333
2,"ARC, Lahore",2025-11-12,265.611111,16.227778,68.888889,3.638889,296.000000,31.522056,74.314944,6125629,1,234.470588,322.333333,270.125000
3,"ARC, Lahore",2025-11-13,234.470588,16.564706,64.764706,3.605882,293.764706,31.522056,74.314944,6125629,1,322.333333,270.125000,231.878261
4,"ARC, Lahore",2025-11-14,322.333333,15.166667,73.866667,1.386667,192.466667,31.522056,74.314944,6125629,1,270.125000,231.878261,283.727273


In [43]:
df = df.sort_values(["date", "name"]).reset_index(drop=True)

In [44]:
features = [
    "pm25",
    "temperature",
    "relativehumidity",
    "wind_speed",
    "wind_direction"
]

In [45]:
split_date = df["date"].quantile(0.8)
train = df[df["date"] <= split_date]
test = df[df["date"] > split_date]

In [46]:
X_test = test[features]
y1_test = test["pm25_day1"]
y2_test = test["pm25_day2"]
y3_test = test["pm25_day3"]

In [47]:
model_day1 = joblib.load("pm25_day1_model.pkl")
model_day2 = joblib.load("pm25_day2_model.pkl")
model_day3 = joblib.load("pm25_day3_model.pkl")

In [48]:
pred_day1 = model_day1.predict(X_test)
pred_day2 = model_day2.predict(X_test)
pred_day3 = model_day3.predict(X_test)

In [49]:
print("========== DAY 1 ==========")
mae = mean_absolute_error(y1_test, pred_day1)
rmse = mean_squared_error(y1_test, pred_day1) ** 0.5
r2 = r2_score(y1_test, pred_day1)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))

========== DAY 1 ==========
MAE : 16.01
RMSE: 21.97
R²  : -0.006


In [50]:
print("========== DAY 2 ==========")
mae = mean_absolute_error(y2_test, pred_day2)
rmse = mean_squared_error(y2_test, pred_day2) ** 0.5
r2 = r2_score(y2_test, pred_day2)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))

========== DAY 2 ==========
MAE : 19.94
RMSE: 26.02
R²  : -0.474


In [51]:
print("========== DAY 3 ==========")
mae = mean_absolute_error(y3_test, pred_day3)
rmse = mean_squared_error(y3_test, pred_day3) ** 0.5
r2 = r2_score(y3_test, pred_day3)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))

========== DAY 3 ==========
MAE : 20.97
RMSE: 28.34
R²  : -0.791


In [52]:
comparison = pd.DataFrame({
    "Actual Day 1": y1_test.values,
    "Predicted Day 1": pred_day1,
    "Actual Day 2": y2_test.values,
    "Predicted Day 2": pred_day2,
    "Actual Day 3": y3_test.values,
    "Predicted Day 3": pred_day3
})

comparison.head(20)

,Actual Day 1,Predicted Day 1,Actual Day 2,Predicted Day 2,Actual Day 3,Predicted Day 3
0,110.854167,93.324300,79.395833,92.029163,66.705263,94.390296
1,51.250000,33.016501,69.125000,35.037969,78.909091,36.476450
2,51.637500,56.838389,43.345833,41.079076,26.620833,52.011174
3,78.043478,87.288713,61.818182,84.462194,51.416667,89.029166
4,72.895833,61.183815,76.787500,60.671373,64.781818,68.866795
5,82.845833,63.655812,81.895833,59.760726,45.981818,71.816356
6,73.909091,71.655818,74.565217,63.617953,53.904762,78.341453
7,78.239130,112.837303,74.812500,104.265024,43.622727,95.086960
8,75.343478,71.253645,74.587500,56.992937,59.238095,67.535207
9,90.045455,94.110909,79.352941,92.347869,60.826087,94.618296


### Reflection

In this notebook, I evaluated three Random Forest regression models designed to forecast PM2.5 concentrations one, two, and three days into the future. Model performance was measured using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the coefficient of determination (R²). The results showed that the Day 1 model achieved the lowest prediction error, while prediction accuracy decreased for Day 2 and Day 3 forecasts. This highlights the difficulty of multi-day air quality forecasting, as PM2.5 levels are influenced by many changing environmental conditions. Although the models have limitations and require further improvement, they establish a baseline forecasting system and provide insight into the challenges of predicting future air quality. Future improvements may include collecting more historical data, adding additional environmental variables, and testing more advanced machine learning approaches.
